# EXEMPLO 1 — Parent-Child Retriever Mínimo
## Aula 6 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Objetivo:** Demonstrar o fluxo essencial do Parent-Child em 3 células de código.

**Tempo:** 15 minutos

## Instalação

In [ ]:
!pip install llama-index llama-index-core llama-index-embeddings-huggingface sentence-transformers --quiet
print('OK')

## Código Principal

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from llama_index.core import Document, StorageContext
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core import VectorStoreIndex
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.retrievers import AutoMergingRetriever
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Documento jurídico de exemplo
texto_lei = """
Lei 13.964/2019 - Pacote Anticrime.
Art. 3o-B. O juiz das garantias e responsavel pelo controle da legalidade da investigacao criminal.
Paragr. 1o O juiz das garantias e competente para:
I - receber a comunicacao imediata da prisao;
II - receber o auto da prisao em flagrante para controle da legalidade;
III - zelar pela observancia dos direitos do preso.
Art. 158-A. Considera-se cadeia de custodia o conjunto de procedimentos utilizados
para manter e documentar a historia cronologica do vestigio coletado em crimes.
A cadeia de custodia se inicia no momento do reconhecimento do elemento como vestigio
e termina com o descarte definitivo do material.
"""

doc = Document(text=texto_lei, metadata={'numero': 'Lei 13.964/2019'})

# Parser hierarquico: pai=256 tokens, filho=64 tokens
parser = HierarchicalNodeParser.from_defaults(chunk_sizes=[256, 64])
all_nodes = parser.get_nodes_from_documents([doc])
leaf_nodes = get_leaf_nodes(all_nodes)

print(f'Total nos: {len(all_nodes)}')
print(f'Nos folha (filhos): {len(leaf_nodes)}')
print(f'Nos pai: {len(all_nodes) - len(leaf_nodes)}')

# Exibir par exemplo
print('\n--- CHUNK FILHO (unidade de busca) ---')
print(leaf_nodes[0].text[:200])
print('\n--- CHUNK PAI (contexto ao LLM) ---')
if leaf_nodes[0].parent_node:
    pai = next((n for n in all_nodes if n.node_id == leaf_nodes[0].parent_node.node_id), None)
    if pai:
        print(pai.text[:400])

## Indexar e Buscar

In [ ]:
# Embedding (fallback rapido)
try:
    embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-m3')
    print('BGE-M3 carregado')
except:
    embed_model = HuggingFaceEmbedding(model_name='sentence-transformers/all-MiniLM-L6-v2')
    print('Fallback: all-MiniLM-L6-v2')

docstore = SimpleDocumentStore()
docstore.add_documents(all_nodes)
storage_context = StorageContext.from_defaults(docstore=docstore)

# Indexar apenas os filhos
index = VectorStoreIndex(leaf_nodes, storage_context=storage_context, embed_model=embed_model)
base_retriever = index.as_retriever(similarity_top_k=4)

retriever = AutoMergingRetriever(base_retriever, storage_context, verbose=True, simple_ratio_thresh=0.3)

# Query
query = 'juiz das garantias recebe comunicacao de prisao?'
nodes = retriever.retrieve(query)

print(f'\nQuery: {query}')
print(f'Nos recuperados: {len(nodes)}')
for n in nodes:
    tipo = 'PAI' if len(n.text) > 150 else 'FILHO'
    print(f'  [{tipo}] {len(n.text)} chars | score: {n.score:.3f}')
    print(f'  Trecho: {n.text[:120]}...')

print('\nCheckpoint: Parent-Child funcionando!')
assert len(nodes) > 0

## Referências (ABNT)

LLAMAINDEX. **Auto-Merging Retriever**. LlamaIndex Documentation, 2024. Disponível em: <https://docs.llamaindex.ai>. Acesso em: abr. 2026.

---
*MBA em RAG & CAG — Aula 6, Exemplo 1*